In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys 

sys.path.append('../utils')
from data_preprocess import * 

In [5]:
dataset_df = pd.read_csv('../data/symptom_data/dataset.csv')
symptom_description_df = pd.read_csv('../data/symptom_data/symptom_Description.csv')
symptom_precaution_df = pd.read_csv('../data/symptom_data/symptom_precaution.csv')
symptom_severity_df = pd.read_csv('../data/symptom_data/Symptom-severity.csv')

In [6]:
dataset_df.head()

,Disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5,Symptom_6,Symptom_7,Symptom_8,Symptom_9,Symptom_10,Symptom_11,Symptom_12,Symptom_13,Symptom_14,Symptom_15,Symptom_16,Symptom_17
0,Fungal infection,itching,skin_rash,nodal_skin_eruptions,dischromic_patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Fungal infection,skin_rash,nodal_skin_eruptions,dischromic_patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Fungal infection,itching,nodal_skin_eruptions,dischromic_patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Fungal infection,itching,skin_rash,dischromic_patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Fungal infection,itching,skin_rash,nodal_skin_eruptions,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
symptom_description_df.head()

,Disease,Description
0,Drug Reaction,An adverse drug reaction (ADR) is an injury ca...
1,Malaria,An infectious disease caused by protozoan para...
2,Allergy,An allergy is an immune system response to a f...
3,Hypothyroidism,"Hypothyroidism, also called underactive thyroi..."
4,Psoriasis,Psoriasis is a common skin disorder that forms...


In [8]:
symptom_precaution_df.head()

,Disease,Precaution_1,Precaution_2,Precaution_3,Precaution_4
0,Drug Reaction,stop irritation,consult nearest hospital,stop taking drug,follow up
1,Malaria,Consult nearest hospital,avoid oily food,avoid non veg food,keep mosquitoes out
2,Allergy,apply calamine,cover area with bandage,NaN,use ice to compress itching
3,Hypothyroidism,reduce stress,exercise,eat healthy,get proper sleep
4,Psoriasis,wash hands with warm soapy water,stop bleeding using pressure,consult doctor,salt baths


In [9]:
symptom_severity_df.head()

,Symptom,weight
0,itching,1
1,skin_rash,3
2,nodal_skin_eruptions,4
3,continuous_sneezing,4
4,shivering,5


In [10]:
# Check for duplicates on each dataset
print("Dataset Duplicates: ", dataset_df.duplicated().sum())
print("Symptom Description Duplicates: ", symptom_description_df.duplicated().sum())
print("Symptom Precause Duplicates: ", symptom_precaution_df.duplicated().sum())
print("Symptom Severity Duplicates: ", symptom_severity_df.duplicated().sum())

Dataset Duplicates:  4616
Symptom Description Duplicates:  0
Symptom Precause Duplicates:  0
Symptom Severity Duplicates:  0


In [11]:
# Check for missing values in each dataset
print("Symptom Description Missing Values: \n", symptom_description_df.isnull().sum())
print("-" * 50)
print("Symptom Precaution Missing Values: \n", symptom_precaution_df.isnull().sum())
print("-" * 50)
print("Symptom Severity Missing Values: \n", symptom_severity_df.isnull().sum())

Symptom Description Missing Values: 
 Disease        0
Description    0
dtype: int64
--------------------------------------------------
Symptom Precaution Missing Values: 
 Disease         0
Precaution_1    0
Precaution_2    0
Precaution_3    1
Precaution_4    1
dtype: int64
--------------------------------------------------
Symptom Severity Missing Values: 
 Symptom    0
weight     0
dtype: int64


In [12]:
# Remove duplicates from dataset.csv 
dataset_df = dataset_df.drop_duplicates()

In [ ]:
dataset_df, symptom_description_df, symptom_precaution_df, symptom_severity_df = process_datasets(
    dataset_df, symptom_description_df, symptom_precaution_df, symptom_severity_df
)

In [ ]:
disease_set = set(dataset_df['Disease'])
dataset = remove_missing(dataset_df, disease_set)


In [ ]:
symptom_precaution_df = remove_missing_precaution(symptom_precaution_df)

In [16]:
severity_lookup = symptom_severity_df.assign(
    Symptom=symptom_severity_df['Symptom']
).set_index('Symptom')['weight'].to_dict()

# Get the weight of a symptom
def get_symptom_weight(symptom):
    if pd.isna(symptom):
        return None 

    if symptom is None:
        return None 
    
    weight = severity_lookup.get(symptom)

    if weight is None:
        raise KeyError(f"Symptom '{symptom}' was not found in Symptom-severity.csv")

    return int(weight)

In [17]:
# Merge each of the four tables into a single dataset
symptom_dataset = list()

for disease in disease_set:

    description = symptom_description_df[symptom_description_df['Disease'] == disease]['Description'].iloc[0]
    precautions = list(symptom_precaution_df[symptom_precaution_df['Disease'] == disease].iloc[0, 1:])
    
    for combination in dataset[disease]:
        
        entry = dict()

        entry['disease'] = disease
        entry['description'] = description 
        entry['symptoms'] = combination
        entry['precaution'] = precautions 
        entry['severity'] = [get_symptom_weight(s) for s in combination]

    
        symptom_dataset.append(entry)
    



In [18]:
import json 
with open("../JSON_data/symptom_dataset.json", "w") as file:
    json.dump(symptom_dataset, file, indent=2)